In [0]:
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

## For Azure AI Vector Store
### In Azure, vector store called as Azure AI Search.
### For this project we are defined the name as Azure-docs-store

##  i created the scopes and stored the api keys end points and target_url for my 2 models


So the real boundary isn't the file — it's your Databricks workspace access controls. The secret scope itself has its own ACLs (who can read/write/manage it), separate from notebook permissions. By default, the creator of a scope gets MANAGE permission, and you can restrict who else can READ it via w.secrets.put_acl(...) (or the CLI/REST equivalent) if you ever share the workspace with collaborators. 


I have deployed 2 models in Azure Foundry 
1. embedding model to convert chunks to embeddings
2. LLM model to generate responses 

##-- I had Already stored the API keys and end points and deployment names in databricks Secrets..

In [0]:

#-- Embedding Model  Credentials------

embedding_model_apikey=dbutils.secrets.get(scope="EmbeddingScope", key="api_key")
embedding_model_target_url=dbutils.secrets.get(scope="EmbeddingScope", key='target_url')


emb_deployment_name=dbutils.secrets.get(scope="EmbeddingScope", key="deployment_name")
emb_version=dbutils.secrets.get(scope="EmbeddingScope", key="emb_version") 

# --  LLM Model CREDENTIALS ------

retrival_model_apikey=dbutils.secrets.get(scope="LLMScope", key="retrival_apikey")
retrival_model_target_url=dbutils.secrets.get(scope="LLMScope", key='retrival_target_url') 

llm_deployment_name=dbutils.secrets.get(scope="LLMScope", key="deployment_name")
llm_version=dbutils.secrets.get(scope="LLMScope", key="version")

#-- Azure AI Vector Store Credentials -----

vector_store_endpoint = dbutils.secrets.get(scope="vector_store_scope", key="azure-search-endpoint")
vector_store_admin_key = dbutils.secrets.get(scope="vector_store_scope", key="azure-search-admin-key")
vector_store_name = dbutils.secrets.get(scope="vector_store_scope", key="azure-vectorstore")



In [0]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=llm_deployment_name,
    base_url=retrival_model_target_url.rsplit('/responses', 1)[0],
    api_key=retrival_model_apikey,
    temperature=0
)

embedding = OpenAIEmbeddings(
    model=emb_deployment_name,
    base_url=embedding_model_target_url.rstrip('/') + '/openai/v1',
    api_key=embedding_model_apikey,
)

## Check the LLM's Connection

In [0]:
sample=llm.invoke('hi i am phani');
print(sample.content)

## Check the embedding model connection

In [0]:
test_vector = embedding.embed_query("Azure Data Factory pipeline")  # Check if the embedding resource is correctly configured and exists.
print("Embedding vector length:", len(test_vector))
print("First 5 values:", test_vector[:5])


## check the Vector Store Connection...



In [0]:
print("Search credentials loaded:", bool(vector_store_endpoint), bool(vector_store_admin_key), bool(vector_store_name))
print(vector_store_name)


## Now the models and vector_store are ready


## Now lets Create the Index for Vector_Store.


In [0]:
 # %pip install azure-search-documents -- commenting because already installed

In [0]:
#dbutils.library.restartPython()


In [0]:
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SearchField,
    SearchFieldDataType,
    SimpleField,
    SearchableField,
    VectorSearch,
    VectorSearchProfile,
    HnswAlgorithmConfiguration,
)

EMBEDDING_DIM = 3072

In [0]:
# -- Credentials that we already Created 

# vector_store_endpoint = dbutils.secrets.get(scope="vector_store_scope", key="azure-search-endpoint")
# vector_store_admin_key = dbutils.secrets.get(scope="vector_store_scope", key="azure-search-admin-key")
# vector_store_name = dbutils.secrets.get(scope="vector_store_scope", key="azure-vectorstore")

# vector_store_name='azure-docs-store';


In [0]:
vector_store_endpoint = dbutils.secrets.get(scope="vector_store_scope", key="azure-search-endpoint")
vectore_store_admin_key = dbutils.secrets.get(scope="vector_store_scope", key="azure-search-admin-key")
vector_store_name = dbutils.secrets.get(scope="vector_store_scope", key="azure-vectorstore")

In [0]:
client=SearchIndexClient(endpoint=vector_store_endpoint, credential=AzureKeyCredential(vector_store_admin_key))
# client was created to manage the indexes for vector store

In [0]:
#-- Define the Schema for index.

# ---- Define the index schema ----
fields = [
    SimpleField(name="id", type=SearchFieldDataType.String, key=True),
    SearchableField(name="content", type=SearchFieldDataType.String),
    SimpleField(name="source", type=SearchFieldDataType.String, filterable=True),
    SimpleField(name="page", type=SearchFieldDataType.Int32, filterable=True),
    SearchField(
        name="content_vector",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        vector_search_dimensions=EMBEDDING_DIM,
        vector_search_profile_name="default-vector-profile",
    ),
]




# Configure the Vector Search 

##-- Define the profile and algorithm then attach the profile to the algorithm 

##-- here we use Hnsw Algo to perform Similarity Search.

In [0]:
# ---- Vector search configuration (HNSW algorithm) ----
vector_search = VectorSearch(
    profiles=[
        VectorSearchProfile(
            name="default-vector-profile",
            algorithm_configuration_name="hnsw",
        )
    ],
    algorithms=[
        HnswAlgorithmConfiguration(name="hnsw")
    ],
)

# ---- Create the index ----

index = SearchIndex(
    name='azure-docs-store',
    fields=fields,
    vector_search=vector_search,
)

result = client.create_or_update_index(index)
print(f"Index Azure-docs-store created/updated successfully.")

## The index part of vector_store was done

### the reason we create this object again was the vector_store name was getting in string format if we directly calling with vectorstorename.add_documents() returning error 

### so we created the new object for vector store here

In [0]:
from langchain_community.vectorstores import AzureSearch

vector_store = AzureSearch(
    azure_search_endpoint=vector_store_endpoint,
    azure_search_key=vector_store_admin_key,
    index_name=vector_store_name,
    embedding_function=embedding.embed_query
)